# 갑자기 멀리 튄 고신뢰 박스를 궤적 이력에 넣을까?

앞선 전체 영상 실험에서 b02의 97번은 천장 쪽 박스를 선택했고 98번은 비어 있었습니다. 같은 b02·c07 영상 4편의 저장된 후보로 이력 갱신만 바꿉니다. 97번과 그 이후의 변화뿐 아니라 417프레임 전체에서 이득과 손실을 확인합니다.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
import numpy as np
import pandas as pd

from wlcv.config import ANNOTATIONS, RUNS, VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.full_video import compare_videos, summarize_video_scope
from wlcv.evaluation.plate_track import BOX, KEY

report = RUNS / "plate_baseline/report"
previous = report / "b02_c07_full_video"
out = report / "high_history_update"
out.mkdir(parents=True, exist_ok=True)
# 15번 노트북에서 저장한 후보를 그대로 사용한다.
candidates = pd.read_csv(previous / "candidates.csv")
review = pd.read_csv(ANNOTATIONS / "b02_c07_plate_review.csv").sort_values(KEY)
saved = pd.read_csv(previous / "per_frame.csv", dtype={"match": "boolean"})
saved = saved[saved.method == "Past link"].reset_index(drop=True)
display(review.groupby(["video_id", "scope", "visibility"]).size().rename("frames"))

## 1. 이력 갱신을 두 방식으로 실행

두 번 모두 현재 프레임의 최고 신뢰도가 0.10 이상이면 같은 박스를 출력하고 2프레임 시계를 갱신합니다. `immediate`는 이 좌표를 항상 이력에 넣습니다. `screened`는 이력이 있을 때만 예상 중심과의 거리를 확인하고, 직전 이력 박스 너비의 0.5배를 벗어나면 좌표 갱신을 건너뜁니다. 두 방식 모두 저신뢰 연결 반지름은 0.5배, 후보 하한은 0.001입니다. 이력이 비었을 때는 고신뢰 박스에서 시작합니다.

In [ ]:
runs = []
low_checks = []
high_checks = []
for method, gate in [("immediate", False), ("screened", True)]:
    detail, checks = compare_videos(review[KEY], candidates, review,
                                    gate_high_history=gate, high_checks=high_checks)
    track = detail[detail.method == "Past link"].reset_index(drop=True).copy()
    if not gate:
        pd.testing.assert_frame_equal(track, saved, check_dtype=False)
    track["method"] = method
    checks["method"] = method
    runs.append(track)
    low_checks.append(checks)

results = pd.concat(runs, ignore_index=True)
checks = pd.concat(low_checks, ignore_index=True)
summary = summarize_video_scope(results)
results.to_csv(out / "per_frame.csv", index=False)
summary.to_csv(out / "summary.csv", index=False)
checks.to_csv(out / "low_checks.csv", index=False)
display(summary[["video_id","scope","method","scored","hits","longest_miss","recovered","wrong_link"]])

## 2. 화면 출력과 이력에 저장된 좌표를 구별하기

`stored=False`인 행은 고신뢰 박스를 출력했지만 이력에는 좌표를 넣지 않은 프레임입니다. 여기 표시되는 IoU는 후보를 선택한 후 검수 박스로 계산합니다. 실제 출력이 달라진 프레임은 별도 표로 확인합니다.

In [ ]:
immediate, screened = runs
high = pd.DataFrame(high_checks).merge(screened[KEY + ["scope","confidence","iou","match"]],
                                       on=KEY, validate="one_to_one")
high.to_csv(out / "high_checks.csv", index=False)
display(high.loc[~high.stored].round(3))

pair = immediate.merge(screened, on=KEY, suffixes=("_before","_after"), validate="one_to_one")
coordinates = ["source", "confidence"] + BOX
before = immediate[coordinates]
after = screened[coordinates]
same = (before.eq(after) | (before.isna() & after.isna())).all(axis=1)
assert same[immediate.source.eq("high")].all()
fields = KEY + ["source_before","source_after","confidence_before","confidence_after",
                "iou_before","iou_after","match_before","match_after"]
changes = pair.loc[~same, fields]
changes.to_csv(out / "changed_frames.csv", index=False)
display(changes.round(4))

## 3. 실제 원판이 빠르게 움직인 프레임

예측과 관계없이 연속 두 장 모두 검수표에서 ‘판독 가능·화면 잘림 없음(readable)’으로 표시된 쌍만 사용합니다. 두 검수 박스 중심 간 거리를 앞 프레임의 검수 박스 너비로 나눈 값이 상위 10%인 프레임을 비교합니다. 이 값은 화면 속 이동량이며 실제 운동 속도(m/s)와 단위가 다릅니다.

In [ ]:
motion = review.copy()
motion["cx"] = (motion.x1 + motion.x2) / 2
motion["cy"] = (motion.y1 + motion.y2) / 2
motion["width"] = motion.x2 - motion.x1
prev = motion.groupby("video_id")[["frame_index","cx","cy","width","visibility"]].shift()
eligible = (motion.frame_index.eq(prev.frame_index + 1) &
            motion.visibility.eq("readable") & prev.visibility.eq("readable"))
distance = np.hypot(motion.cx - prev.cx, motion.cy - prev.cy)
motion["movement_in_box_widths"] = (distance / prev.width).where(eligible)
limit = motion.movement_in_box_widths.quantile(.90)
fast = motion.loc[motion.movement_in_box_widths.ge(limit), KEY + ["movement_in_box_widths"]]
fast = fast.merge(pair[fields], on=KEY, validate="one_to_one")
fast["lost"] = fast.match_before & ~fast.match_after
fast["gained"] = ~fast.match_before & fast.match_after
fast.to_csv(out / "fast_frames.csv", index=False)
print(f"비교 가능한 연속 프레임 쌍 {eligible.sum()}개, 상위 10% 경계 {limit:.3f} 박스 너비/프레임")
display(fast.groupby("video_id").agg(frames=("frame_index","size"), before=("match_before","sum"),
                                     after=("match_after","sum"), lost=("lost","sum"), gained=("gained","sum")))

## 4. 같은 프레임에 두 방법의 박스와 예상 위치 그리기

노란 실선은 검수 박스, 청록 점선은 출력 박스입니다. 저신뢰 후보를 검사한 프레임에는 예상 중심과 허용 반지름을 분홍색으로 그립니다. 97번에서 고신뢰 출력은 같은지, 98번에서 원의 위치가 어떻게 달라졌는지 확인합니다. 100·101번은 선택한 박스가 원판 전체를 얼마나 감싸는지도 살펴봅니다.

In [ ]:
video = "b02_sn_h_02"
numbers = [97, 98, 100, 101]
images = {n: read_rgb(VIDEOS / f"{video}.mp4", n) for n in numbers}
fig, axes = plt.subplots(2, 4, figsize=(11, 8.7), layout="constrained")
for row_axes, table in zip(axes, runs):
    method = table.method.iloc[0]
    for ax, n in zip(row_axes, numbers):
        frame = images[n]
        row = table[(table.video_id == video) & (table.frame_index == n)].iloc[0]
        ax.imshow(frame)
        for coords, color, style in [(row[["gt_"+k for k in BOX]], "#ffd23c", "-"),
                                     (row[BOX], "#14eff5", "--")]:
            if coords.notna().all():
                x1, y1, x2, y2 = coords
                ax.add_patch(Rectangle((x1,y1), x2-x1, y2-y1, fill=False, ec=color, ls=style, lw=2))
        probe = checks[(checks.method == method) & (checks.video_id == video) & (checks.frame_index == n)]
        if not probe.empty:
            p = probe.iloc[0]
            ax.add_patch(Circle((p.expected_x,p.expected_y), p.radius_px, fill=False, ec="#ff64be", lw=2))
            ax.plot(p.expected_x,p.expected_y,"+",color="#ff64be",ms=9)
        # 공통 여백에 화면 밖으로 나간 예상 중심도 표시한다.
        ax.set_xlim(-170, frame.shape[1]+220)
        ax.set_ylim(frame.shape[0], 0)
        ax.set_facecolor("#eee")
        ax.set_xticks([])
        ax.set_yticks([])
        label = "No output" if pd.isna(row.iou) else f"IoU {row.iou:.3f}"
        ax.set_title(f"{method} | f{n}\n{label}", fontsize=10)
fig.savefig(out / "history_examples.png", dpi=140)
plt.show()

## 5. 연속 미탐이 실제로 줄었는지 확인하기

IoU 0.50 이상이 정탐입니다. 출력이 비어 있는 프레임은 그래프 하단에 따로 표시합니다. 박스를 새로 출력했어도 IoU가 기준 아래이면 미탐 길이는 계속 이어집니다.

In [ ]:
fig, ax = plt.subplots(figsize=(9,3.8), layout="constrained")
for table, offset, color, marker in zip(runs, [-.10,.10], ["#345997","#b44323"], ["o","D"]):
    data = table[(table.video_id == video) & table.frame_index.between(92,105)]
    ax.scatter(data.frame_index + offset, data.iou.fillna(-.14), color=color, marker=marker,
               label=table.method.iloc[0], s=44)
ax.axhline(.5, color="#454545", ls="--", lw=1, label="IoU 0.50")
ax.set(xticks=range(92,106), yticks=[-.14,0,.25,.5,.75,1],
       yticklabels=["No output","0",".25",".50",".75","1"], xlabel="Frame",
       ylabel="IoU with reviewed plate", title="b02_sn_h_02 | Paired frame outcomes")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(.5,-.16))
fig.savefig(out / "paired_iou.png", dpi=145)
plt.show()